# T7 · Reconcile
Subtasks: thirteen checks, the province × mode matrix, the last question. All in this one notebook.

In [1]:
import pandas as pd, numpy as np, os
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 100)

p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p, parse_dates=["ShipDate"])
assert len(src) == 34 and src.WeightKG.sum() == 578_650 and src.CostPKR.sum() == 4_808_000

# ---------- Dimensions from T4 / T5 (rebuilt compactly so this notebook runs on its own) ----------
city_key = {"Karachi":1,"Gwadar":2,"Lahore":3,"Multan":4,"Islamabad":5,"Quetta":6,"Faisalabad":7,
            "Rawalpindi":8,"Sialkot":9,"Peshawar":10,"Hyderabad":11,"Sukkur":12,"Gujranwala":13,"Abbottabad":14}
alias_city = {"Karachi Port":"Karachi","Port Qasim":"Karachi","Karachi Cantt":"Karachi","Pipri Marshalling Yard":"Karachi","Karachi":"Karachi",
              "Gwadar Port":"Gwadar","Lahore":"Lahore","Lahore Dry Port":"Lahore","Multan":"Multan","Islamabad":"Islamabad",
              "Islamabad Dry Port":"Islamabad","Quetta":"Quetta","Faisalabad":"Faisalabad","Faisalabad Dry Port":"Faisalabad",
              "Rawalpindi":"Rawalpindi","Sialkot Dry Port":"Sialkot","Sialkot":"Sialkot","Peshawar":"Peshawar","Peshawar Cantt":"Peshawar",
              "Hyderabad":"Hyderabad","Sukkur":"Sukkur","Gujranwala":"Gujranwala","Abbottabad":"Abbottabad"}
province = {"Karachi":"Sindh","Hyderabad":"Sindh","Sukkur":"Sindh","Gwadar":"Balochistan","Quetta":"Balochistan",
            "Lahore":"Punjab","Faisalabad":"Punjab","Multan":"Punjab","Sialkot":"Punjab","Gujranwala":"Punjab","Rawalpindi":"Punjab",
            "Islamabad":"Islamabad Capital Territory","Peshawar":"Khyber Pakhtunkhwa","Abbottabad":"Khyber Pakhtunkhwa"}

# DimCity: one row per city + alias table used for the lookup + the -1 row
DimCity = pd.DataFrame([(k, c, province[c]) for c, k in city_key.items()] + [(-1, "Unknown", "Unknown")],
                       columns=["CityKey", "CityName", "Province"])
CityAlias = pd.DataFrame({"RawString": list(alias_city), "CityKey": [city_key[c] for c in alias_city.values()]})

# DimRoute: conform first, then dedupe on the key pair (first-appearance order), plus -1
pairs = pd.DataFrame({"OriginCityKey": src.Origin.map(dict(zip(CityAlias.RawString, CityAlias.CityKey))),
                      "DestCityKey":   src.Destination.map(dict(zip(CityAlias.RawString, CityAlias.CityKey)))}).drop_duplicates().reset_index(drop=True)
DimRoute = pd.concat([pairs.assign(RouteKey=range(1, len(pairs) + 1)),
                      pd.DataFrame([{"RouteKey": -1, "OriginCityKey": -1, "DestCityKey": -1}])], ignore_index=True)

DimMode   = pd.DataFrame([(1, "Marine"), (2, "Rail"), (3, "Road"), (-1, "Unknown")], columns=["ModeKey", "ModeName"])
DimStatus = pd.DataFrame([(1, "Delivered"), (2, "In Transit"), (3, "Delayed"), (-1, "Unknown")], columns=["StatusKey", "StatusName"])
DimDate   = pd.DataFrame({"FullDate": pd.date_range("2026-02-01", "2026-02-28")})
DimDate["DateKey"] = DimDate.FullDate.dt.strftime("%Y%m%d").astype(int)
assert len(DimCity) == 15 and len(DimRoute) == 21 and len(DimDate) == 28

In [2]:
def build_fact(df):
    """Swap every text column for an integer key by LOOKUP (left join + -1), never typed digits."""
    x = df.copy()
    alias = CityAlias.set_index("RawString").CityKey
    x["OriginCityKey"] = x.Origin.map(alias)                       # NaN when the string is unmapped
    x["DestCityKey"]   = x.Destination.map(alias)
    x = x.merge(DimRoute[DimRoute.RouteKey != -1][["RouteKey", "OriginCityKey", "DestCityKey"]],
                on=["OriginCityKey", "DestCityKey"], how="left")
    x["ModeKey"]   = x.SourceSystem.map(DimMode.set_index("ModeName").ModeKey)
    x["StatusKey"] = x.Status.map(DimStatus.set_index("StatusName").StatusKey)
    x["DateKey"]   = x.ShipDate.dt.strftime("%Y%m%d").astype(int)
    x["DateKey"]   = x.DateKey.where(x.DateKey.isin(DimDate.DateKey))
    keys = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey"]
    x[keys] = x[keys].fillna(-1).astype(int)                       # the IFERROR(..., -1) step
    x["ShipmentCount"] = 1
    return x.rename(columns={"CostPKR": "FreightCostPKR"})[keys + ["ShipmentID", "WeightKG", "FreightCostPKR", "ShipmentCount"]]

FactShipment = build_fact(src)
FactShipment

,DateKey,OriginCityKey,DestCityKey,RouteKey,ModeKey,StatusKey,ShipmentID,WeightKG,FreightCostPKR,ShipmentCount
0,20260201,1,3,1,1,1,MR-1001,18000,142000,1
1,20260202,1,7,2,1,1,MR-1002,22500,171000,1
2,20260203,1,3,1,1,2,MR-1003,15000,119000,1
3,20260204,2,6,3,1,1,MR-1004,20000,188000,1
4,20260205,1,4,4,1,1,MR-1005,16500,128000,1
5,20260206,1,9,5,1,3,MR-1006,19000,154000,1
6,20260207,1,3,1,1,1,MR-1007,14000,109000,1
7,20260209,1,10,6,1,2,MR-1008,21000,197000,1
8,20260210,2,1,7,1,1,MR-1009,17500,98000,1
9,20260211,1,5,8,1,1,MR-1010,15500,139000,1


## The thirteen checks

In [3]:
keys = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey"]
real = lambda d, k: int((d[k] != -1).sum())

def province_mode(fact, city_col="OriginCityKey"):
    """Verification join through the dimensions (province and mode name are NOT stored in the fact)."""
    return (fact.merge(DimCity[["CityKey", "Province"]], left_on=city_col, right_on="CityKey")
                .merge(DimMode, on="ModeKey")
                .pivot_table(index="Province", columns="ModeName", values="WeightKG", aggfunc="sum", fill_value=0))

pm = province_mode(FactShipment)
checks = pd.DataFrame([
 ( 1, "Row count of FactShipment",                34,        len(FactShipment)),
 ( 2, "Grand total, WeightKG",                    578_650,   int(FactShipment.WeightKG.sum())),
 ( 3, "Grand total, FreightCostPKR",              4_808_000, int(FactShipment.FreightCostPKR.sum())),
 ( 4, "Shipment legs (SUM of Count)",             34,        int(FactShipment.ShipmentCount.sum())),
 ( 5, "Distinct ShipmentID values",               34,        FactShipment.ShipmentID.nunique()),
 ( 6, "Rows in DimCity (excluding Unknown)",      14,        real(DimCity, "CityKey")),
 ( 7, "Rows in DimRoute (excluding Unknown)",     20,        real(DimRoute, "RouteKey")),
 ( 8, "Rows in DimDate",                          28,        len(DimDate)),
 ( 9, "WeightKG for Sindh / Marine",              169_000,   int(pm.loc["Sindh", "Marine"])),
 (10, "WeightKG for Punjab / Road",               44_100,    int(pm.loc["Punjab", "Road"])),
 (11, "WeightKG for Sindh, all modes",            484_350,   int(pm.loc["Sindh"].sum())),
 (12, "Fact rows pointing at an Unknown key (-1)", 0,        int((FactShipment[keys] == -1).any(axis=1).sum())),
 (13, "Cost per KG, grand total",                 8.31,      round(FactShipment.FreightCostPKR.sum() / FactShipment.WeightKG.sum(), 2)),
], columns=["#", "Check", "Expected", "Your result"])
checks["Pass?"] = np.where(checks.Expected == checks["Your result"], "pass", "fix")
print("CHECKS PASSING:", int((checks["Pass?"] == "pass").sum()), "/ 13")
assert (checks["Pass?"] == "pass").all(), "a red cell loses all 8 marks"
checks

CHECKS PASSING: 13 / 13


,#,Check,Expected,Your result,Pass?
0,1,Row count of FactShipment,34.00,34.00,pass
1,2,"Grand total, WeightKG",578650.00,578650.00,pass
2,3,"Grand total, FreightCostPKR",4808000.00,4808000.00,pass
3,4,Shipment legs (SUM of Count),34.00,34.00,pass
4,5,Distinct ShipmentID values,34.00,34.00,pass
5,6,Rows in DimCity (excluding Unknown),14.00,14.00,pass
6,7,Rows in DimRoute (excluding Unknown),20.00,20.00,pass
7,8,Rows in DimDate,28.00,28.00,pass
8,9,WeightKG for Sindh / Marine,169000.00,169000.00,pass
9,10,WeightKG for Punjab / Road,44100.00,44100.00,pass


### The two diagnostic checks: what they would read if the mistake were made

In [4]:
# #9: Gwadar Port mapped to Karachi (the classic conforming error)
bad_alias = CityAlias.assign(CityKey=np.where(CityAlias.RawString == "Gwadar Port", city_key["Karachi"], CityAlias.CityKey))
x = src.assign(OriginCityKey=src.Origin.map(bad_alias.set_index("RawString").CityKey)).merge(DimCity[["CityKey", "Province"]], left_on="OriginCityKey", right_on="CityKey")
print("#9  with Gwadar mapped to Karachi :", int(x[(x.Province == "Sindh") & (x.SourceSystem == "Marine")].WeightKG.sum()), "(correct 169,000)")
# #13: the T3 mistake, averaging the per-row ratio
print("#13 with the average of a ratio    :", round((src.CostPKR / src.WeightKG).mean(), 2), "(correct 8.31)")

#9  with Gwadar mapped to Karachi : 206500 (correct 169,000)
#13 with the average of a ratio    : 8.06 (correct 8.31)


## The matrix: tonnage by origin province and mode

In [5]:
order = ["Sindh", "Punjab", "Balochistan", "Khyber Pakhtunkhwa", "Islamabad Capital Territory"]
cols  = ["Marine", "Rail", "Road"]
M = pm.reindex(index=order, columns=cols, fill_value=0)
M["Total"] = M.sum(axis=1)
M.loc["Grand total"] = M.sum()
M.index = M.index.str.replace("Islamabad Capital Territory", "Islamabad ICT")
M.loc["Reconciles to 578,650?"] = ["", "", "", "yes" if M.loc["Grand total", "Total"] == 578_650 else "NO"]
assert M.loc["Grand total", "Total"] == 578_650
M

ModeName,Marine,Rail,Road,Total
Province,,,,
Sindh,169000,295250,20100,484350
Punjab,0,0,44100,44100
Balochistan,37500,0,6600,44100
Khyber Pakhtunkhwa,0,0,0,0
Islamabad ICT,0,0,6100,6100
Grand total,206500,295250,76900,578650
"Reconciles to 578,650?",,,,yes


In [6]:
# The zero row (Khyber Pakhtunkhwa) is correct for ORIGIN: rebuild on DestCityKey, the other role of DimCity
D = province_mode(FactShipment, "DestCityKey").reindex(index=order, columns=cols, fill_value=0)
D["Total"] = D.sum(axis=1)
D.loc["Grand total"] = D.sum()
D.index = D.index.str.replace("Islamabad Capital Territory", "Islamabad ICT")
assert D.loc["Grand total", "Total"] == 578_650 and D.loc["Khyber Pakhtunkhwa", "Total"] == 61_900
D

ModeName,Marine,Rail,Road,Total
Province,,,,
Sindh,31000,28250,26700,85950
Punjab,119000,238000,31800,388800
Balochistan,20000,0,0,20000
Khyber Pakhtunkhwa,21000,29000,11900,61900
Islamabad ICT,15500,0,6500,22000
Grand total,206500,295250,76900,578650


**Reading the zero:** the Khyber Pakhtunkhwa row is 0 because the matrix is built on `OriginCityKey`: nothing left Peshawar or Abbottabad in February. The number is correct, but "KP is not part of our network" would be a false conclusion: 61,900 kg moved *into* KP. Rebuilt on `DestCityKey` the same `DimCity` shows 61,900 kg. `DimCity` is a role-playing dimension, and the active relationship decides which question is answered (in Power BI, `USERELATIONSHIP` switches to the other).

## The last question

In [7]:
# Corrupt the model in a way no check looks at: swap the Delivered and Delayed keys on the fact
swap = {1: 3, 3: 1, 2: 2}
bad = FactShipment.assign(StatusKey=FactShipment.StatusKey.map(swap))
bm = province_mode(bad)
still = [len(bad) == 34, bad.WeightKG.sum() == 578_650, bad.FreightCostPKR.sum() == 4_808_000, bad.ShipmentCount.sum() == 34,
         bad.ShipmentID.nunique() == 34, bm.loc["Sindh", "Marine"] == 169_000, bm.loc["Punjab", "Road"] == 44_100,
         bm.loc["Sindh"].sum() == 484_350, int((bad[keys] == -1).any(axis=1).sum()) == 0]
print("Every check that could be affected still passes:", all(still))

delayed = lambda f: int((f.StatusKey == 3).sum())
print("Delayed legs in the source:", int((src.Status == "Delayed").sum()), "| Delayed legs in the corrupted fact:", delayed(bad))

Every check that could be affected still passes: True
Delayed legs in the source: 4 | Delayed legs in the corrupted fact: 22


**One thing no check would catch:** a *valid but wrong* key. All thirteen checks test counts, totals and a few origin-province weights, and none tests `StatusKey`, `DestCityKey`, `RouteKey` or `DateKey` against the source. If the Delivered and Delayed keys were swapped (or two destination cities mapped to each other, or a route pointing at the wrong city pair), every check above would still read pass, as the test shows: the row count, kilograms and PKR totals do not move, and no key is -1, yet every status report would be wrong. The `-1` row only catches keys that fail to resolve, not keys that resolve to the wrong member. The fix is a content check, not a total check: compare a count or weight by status, destination province and route in the fact (looked up through the dimension) against the same cut computed directly from the source text. The destination matrix above (KP = 61,900 kg) is one such check.